# 03 - Feature Engineering

Lag 1/2/3/12, rolling mean 3/6 (hanya data sebelum bulan target), per pasar x komoditas. Ditambah fitur musiman: penanda Ramadan dan Idul Fitri (dari tanggal, bukan harga), serta posisi bulan (sin/cos). `geo_id` tidak dipakai sebagai fitur.


In [1]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd, numpy as np
from IPython.display import Image, display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

In [2]:
from src.data_loader import load_long
from src.feature_engineering import build_features, FEATURES
long = load_long()
feat = build_features(long)
d = feat[feat.target.notna() & feat.eligible]
print(len(feat), len(d))
d.head()

371406 104460


,date,geo_id,commodity,target,price_lag_1,price_lag_2,price_lag_3,price_lag_12,rolling_mean_3,rolling_mean_6,n_obs_last6,mom_1,dev_roll3,dev_roll6,year,month_sin,month_cos,ramadan_share,lebaran_bulan,pra_lebaran,adm1_name,eligible
159,2020-04-01,gid_-1000001093100000,eggs,25500.00,23324.00,23770.69,26633.33,NaN,24576.006667,24576.006667,3.0,-0.018792,-0.050944,-0.050944,2020,8.660254e-01,-0.500000,0.233333,0.0,1.0,KALIMANTAN BARAT,True
160,2020-05-01,gid_-1000001093100000,eggs,25742.86,25500.00,23324.00,23770.69,NaN,24198.230000,24807.005000,4.0,0.093294,0.053796,0.027935,2020,5.000000e-01,-0.866025,0.766667,1.0,0.0,KALIMANTAN BARAT,True
161,2020-06-01,gid_-1000001093100000,eggs,28753.45,25742.86,25500.00,23324.00,NaN,24855.620000,24994.176000,5.0,0.009524,0.035696,0.029954,2020,1.224647e-16,-1.000000,0.000000,0.0,0.0,KALIMANTAN BARAT,True
162,2020-07-01,gid_-1000001093100000,eggs,29183.33,28753.45,25742.86,25500.00,NaN,26665.436667,25620.721667,6.0,0.116949,0.078304,0.122273,2020,-5.000000e-01,-0.866025,0.000000,0.0,0.0,KALIMANTAN BARAT,True
163,2020-08-01,gid_-1000001093100000,eggs,28834.62,29183.33,28753.45,25742.86,NaN,27893.213333,26045.721667,6.0,0.014951,0.046252,0.120465,2020,-8.660254e-01,-0.500000,0.000000,0.0,0.0,KALIMANTAN BARAT,True


In [3]:
# Uji kebocoran: lag_1 harus sama dengan target bulan sebelumnya pada grup yang sama
g = feat[(feat.geo_id==feat.geo_id.iloc[5000]) & (feat.commodity==feat.commodity.iloc[5000])].reset_index(drop=True)
m = g.target.iloc[:-1].notna().values
print('lag_1 == target bulan lalu:', (g.price_lag_1.iloc[1:].values[m] == g.target.iloc[:-1].values[m]).all())

lag_1 == target bulan lalu: True


## Fitur musiman

Bulan Ramadan, Idul Fitri, dan pra-Idul Fitri per tahun. Tanggal ada di `src/feature_engineering.py`.

In [4]:
from src.feature_engineering import seasonal_features, RAMADAN_START, EID_FITRI
print('Fitur numerik:', FEATURES)
tgl = pd.Series(pd.date_range('2024-01-01', '2026-06-01', freq='MS'))
s = seasonal_features(tgl).assign(bulan=tgl.dt.strftime('%Y-%m'))
s[s[['ramadan_share','lebaran_bulan','pra_lebaran']].sum(axis=1) > 0]

Fitur numerik: ['commodity', 'adm1_name', 'year', 'month_sin', 'month_cos', 'ramadan_share', 'lebaran_bulan', 'pra_lebaran', 'price_lag_1', 'price_lag_2', 'price_lag_3', 'price_lag_12', 'rolling_mean_3', 'rolling_mean_6', 'n_obs_last6', 'mom_1', 'dev_roll3', 'dev_roll6']


,month_sin,month_cos,ramadan_share,lebaran_bulan,pra_lebaran,bulan
2,1.000000,6.123234e-17,0.700000,0.0,1.0,2024-03
3,0.866025,-5.000000e-01,0.300000,1.0,0.0,2024-04
13,0.866025,5.000000e-01,0.000000,0.0,1.0,2025-02
14,1.000000,6.123234e-17,1.000000,1.0,0.0,2025-03
25,0.866025,5.000000e-01,0.333333,0.0,1.0,2026-02
26,1.000000,6.123234e-17,0.666667,1.0,0.0,2026-03


In [5]:
d.groupby(d.date.dt.year).size()

date
2016     2739
2017        3
2020    12412
2021    17123
2022    17358
2023    17102
2024     9925
2025    17025
2026    10773
dtype: int64